# 🧠 Week 9 Lab — Student Version
## SVM: Finding the Widest Street

Last week we learned KNN: a non-parametric classifier that stores the entire training set
and classifies by proximity. It achieved perfect EMG binary diagnosis but suffered from
the curse of dimensionality.

This week we implement **Support Vector Machines** — a fundamentally different approach
that finds the maximum-margin boundary between classes. Instead of asking "who is this
trial closest to?" the SVM asks "where is the widest gap between classes?"

**What you'll implement:**
- Visualize the maximum-margin boundary and its support vectors
- Tune the C hyperparameter and observe the margin–accuracy tradeoff
- Compare linear and RBF kernels on our reaching dataset
- Test the kernel trick on nonlinearly separable data
- Build confusion matrices and per-direction F1 scores
- Measure robustness to electrode drift and computational cost
- Update the running comparison table with SVM results

**Difficulty guide:** 🟢 Core (do first) → 🟡 Intermediate → 🔴 Advanced

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload week8_data.pkl

In [ ]:
# Load the dataset
with open('week8_data.pkl', 'rb') as f:
    data = pickle.load(f)

# Unpack
X_raw = data['X_raw']            # (480, 6) EMG features
neural_rates = data['neural_rates']  # (480, 80) neural firing rates
targets = data['targets']        # (480,) direction labels (0°, 45°, ..., 315°)
subjects = data['subjects']      # (480,) subject IDs (0–19)
joint_angles = data['joint_angles']  # (480, 2) shoulder & elbow angles

# Build binary labels: subjects 0–9 = healthy, 10–19 = impaired
labels_str = data['labels']      # (480,) 'healthy' or 'impaired'
group_binary = (labels_str == 'impaired').astype(int)  # 0=healthy, 1=impaired

# Useful constants
dir_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
logo = LeaveOneGroupOut()

print(f"Dataset: {X_raw.shape[0]} trials, {len(np.unique(subjects))} subjects, "
      f"{len(np.unique(targets))} directions")
print(f"EMG features: {X_raw.shape[1]}, Neural features: {neural_rates.shape[1]}")
print(f"Binary labels: {(group_binary==0).sum()} healthy, {(group_binary==1).sum()} impaired")

---

## 🟢 Part 1: From Parameters to Margins (Lecture §1)

The lecture argued that when multiple boundaries can separate the data, we should
pick the one with the **widest margin** — the largest gap between the boundary and
the nearest training trial. This gives the most robustness to electrode drift and
inter-session variability.

In this part, you'll visualize the SVM margin and its support vectors on our
neural reaching data, and contrast it with the boundaries from prior weeks.

### Exercise 1.1: Visualize the SVM margin on 0° vs 45° (Lecture Figure 1)

**Learning objective:** See the maximum-margin boundary, the margin width, and the
support vectors on real data.

Take two adjacent reach directions (0° and 45°) in neural PCA space. Fit a linear
SVM and plot the decision boundary, margin lines, and support vectors.

In [ ]:
# Exercise 1.1: SVM margin on 0° vs 45° neural data
# Step 1: Select 0° and 45° trials, PCA to 2D
mask = np.isin(targets, [0, 1])  # 0=0°, 1=45°
X_2dir = neural_rates[mask]
y_2dir = targets[mask]

sc = StandardScaler()
pca = PCA(n_components=2)
X_2d = pca.fit_transform(sc.fit_transform(X_2dir))

# Step 2: Fit linear SVM
svm = SVC(kernel='linear', C=1.0)
svm.fit(X_2d, y_2dir)

# Step 3: Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Multiple possible boundaries (simulate a few random separators)
ax = axes[0]
ax.scatter(X_2d[y_2dir == 0, 0], X_2d[y_2dir == 0, 1], c='steelblue', label='0°', alpha=0.6)
ax.scatter(X_2d[y_2dir == 1, 0], X_2d[y_2dir == 1, 1], c='coral', label='45°', alpha=0.6)

# TODO: Draw 3-4 random separating lines through the gap to show that many boundaries work
# Hint: use ax.plot() with different slopes passing between the two clusters

ax.set_title('(A) Many Valid Separating Boundaries')
ax.set_xlabel('Neural PC1')
ax.set_ylabel('Neural PC2')
ax.legend()

# Panel B: The SVM solution
ax = axes[1]
ax.scatter(X_2d[y_2dir == 0, 0], X_2d[y_2dir == 0, 1], c='steelblue', label='0°', alpha=0.6)
ax.scatter(X_2d[y_2dir == 1, 0], X_2d[y_2dir == 1, 1], c='coral', label='45°', alpha=0.6)

# TODO: Plot the decision boundary and margin lines
# Hint: Use svm.coef_ and svm.intercept_ to get w and b
# Decision boundary: w[0]*x + w[1]*y + b = 0
# Margin lines: w[0]*x + w[1]*y + b = ±1

# TODO: Circle the support vectors using svm.support_vectors_

# TODO: Compute and display the margin width = 2 / ||w||
# w = svm.coef_[0]; margin = 2 / np.linalg.norm(w)

ax.set_title('(B) SVM: Maximum Margin Boundary')
ax.set_xlabel('Neural PC1')
ax.set_ylabel('Neural PC2')
ax.legend()
plt.tight_layout()
plt.show()

# Print key numbers
w = svm.coef_[0]
print(f"Weight vector: w = [{w[0]:.3f}, {w[1]:.3f}]")
print(f"||w|| = {np.linalg.norm(w):.3f}")
print(f"Margin width = 2/||w|| = {2/np.linalg.norm(w):.3f}")
print(f"Support vectors: {len(svm.support_vectors_)} out of {len(y_2dir)} trials")

### Exercise 1.2: Compare prior classifiers' boundaries (Lecture Figure 2)

**Learning objective:** See that LR, Naive Bayes, and KNN all produce valid boundaries
but none of them explicitly maximise the margin.

Fit LR, NB, and KNN on the full 8-direction neural data (PCA to 2D) and plot their
decision boundaries side by side.

In [ ]:
# Exercise 1.2: Prior classifiers' boundaries in neural PCA space
sc = StandardScaler()
pca = PCA(n_components=2)
X_pca2 = pca.fit_transform(sc.fit_transform(neural_rates))

classifiers = {
    'Logistic Regression': LogisticRegression(C=10, max_iter=2000),
    'Naive Bayes': GaussianNB(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmap8 = plt.cm.hsv(np.linspace(0, 0.9, 8))

for ax, (name, clf) in zip(axes, classifiers.items()):
    clf.fit(X_pca2, targets)

    # TODO: Create meshgrid and plot decision regions using contourf
    # Hint: xx, yy = np.meshgrid(np.linspace(...), np.linspace(...))
    # Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    # TODO: Scatter training points colored by direction
    
    ax.set_title(name)
    ax.set_xlabel('Neural PC1')
    ax.set_ylabel('Neural PC2')

plt.suptitle('Prior Classifiers: None Maximise the Margin', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

---

## 🟢 Part 2: Testing SVM on Our Data (Lecture §2)

The lecture introduced C as the hyperparameter that controls the trade-off between
a wide margin and correct classification. C is a *hyperparameter* — a choice we make
before training, unlike *parameters* (w, b) that the SVM learns from data.

Here you'll sweep C, observe its effect on the margin and support vector count, and
compare linear and RBF SVMs against all prior methods.

### Exercise 2.1: C sweep — margin vs accuracy (Lecture Figure 3)

**Learning objective:** See that small C gives a wide margin but low accuracy, while
large C gives a narrow margin but high accuracy. The optimal C balances the two.

Sweep C across several orders of magnitude for linear SVM on neural direction decoding.

In [ ]:
# Exercise 2.1: C sweep for linear SVM
C_values = [0.001, 0.01, 0.1, 1, 10, 100]

accs = []
n_svs = []

for C in C_values:
    pipe = Pipeline([('s', StandardScaler()),
                     ('svm', SVC(kernel='linear', C=C))])
    
    # TODO: Run LOSO cross-validation for neural direction decoding
    # Hint: scores = cross_val_score(pipe, neural_rates, targets, cv=logo, groups=subjects)
    
    # TODO: Fit on all data to count support vectors
    # Hint: pipe.fit(neural_rates, targets); n_sv = pipe['svm'].n_support_.sum()
    
    pass  # Replace with your code

# TODO: Plot accuracy vs C (semilog x-axis)
# TODO: On a second y-axis, plot the number of support vectors vs C
# Hint: ax2 = ax.twinx()

plt.show()

### Exercise 2.2: Support vector count vs C

**Learning objective:** Understand how C controls the number of support vectors.
At C = 0.001, every trial is a support vector (maximum slack). As C increases,
fewer trials sit on or inside the margin.

Print the support vector count at each C value and interpret the result.

In [ ]:
# Exercise 2.2: Support vector count at each C
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    pipe = Pipeline([('s', StandardScaler()),
                     ('svm', SVC(kernel='linear', C=C))])
    pipe.fit(neural_rates, targets)
    
    n_sv = pipe['svm'].n_support_.sum()
    # TODO: Print C, number of SVs, and percentage of total trials
    # Why does the count stabilize above C=0.1?

### Exercise 2.3: Five-method comparison (Lecture Figure 4)

**Learning objective:** Compare Linear SVM and RBF SVM against Naive Bayes, Logistic
Regression, and KNN on all four tasks (direction × binary) × (EMG × neural).

Use C = 1 for linear SVM, and C = 10 with γ = 'scale' for RBF SVM.

In [ ]:
# Exercise 2.3: Five-method comparison
methods = {
    'Naive Bayes': GaussianNB(),
    'LR (C=10)': LogisticRegression(C=10, max_iter=2000),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Linear SVM': SVC(kernel='linear', C=1),
    'RBF SVM': SVC(kernel='rbf', C=10, gamma='scale'),
}

tasks = {
    'EMG Direction': (X_raw, targets),
    'Neural Direction': (neural_rates, targets),
    'EMG Binary': (X_raw, group_binary),
    'Neural Binary': (neural_rates, group_binary),
}

results = {}
for tname, (X, y) in tasks.items():
    results[tname] = {}
    for mname, clf in methods.items():
        pipe = Pipeline([('s', StandardScaler()), ('clf', clf)])
        # TODO: Run LOSO cross-validation
        # TODO: Store mean accuracy in results[tname][mname]
        pass

# TODO: Create grouped bar chart — direction tasks on the left, binary on the right
# TODO: Add chance lines (12.5% for direction, 50% for binary)

plt.show()

### 🤔 Thought Exercise
The lecture showed that C = 0.001 works for 8-direction decoding but gives 0%
accuracy on binary diagnosis. Why? Think about what happens when the margin penalty
dominates so completely that the SVM allows every trial to violate the boundary.

In [ ]:
# Your reflection:


---

## 🟡 Part 3: The Kernel Trick (Lecture §3)

The lecture showed that the linear SVM achieved only 70% on binary diagnosis because
healthy-vs-impaired is a nonlinear distinction. The kernel trick lets the SVM operate
in a higher-dimensional feature space where linear separation becomes possible,
without ever computing the mapped coordinates.

Here you'll visualize the kernel trick on a toy example and compare kernels on our data.

### Exercise 3.1: Concentric rings — linear fails, RBF succeeds (Lecture Figure 6)

**Learning objective:** See the kernel trick in action on a problem that is impossible
for any linear classifier.

In [ ]:
# Exercise 3.1: Concentric rings
from sklearn.datasets import make_circles
X_rings, y_rings = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, kernel, title in [(axes[0], 'linear', 'Linear SVM — Fails'),
                           (axes[1], 'rbf', 'RBF SVM — Succeeds')]:
    svm = SVC(kernel=kernel, C=10, gamma='scale')
    svm.fit(X_rings, y_rings)
    
    # TODO: Create meshgrid and plot decision boundary with contourf
    # TODO: Scatter data points, color by class
    # TODO: Circle support vectors
    
    acc = svm.score(X_rings, y_rings)
    ax.set_title(f'{title}\nAccuracy: {acc:.1%}')

plt.suptitle('The Kernel Trick: When Linear Fails', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 3.2: Kernel comparison on neural direction decoding (Lecture Figure 5)

**Learning objective:** Compare linear, polynomial, and RBF kernels on our data.
For direction decoding, the linear kernel should do well because cosine tuning
creates approximately linear class boundaries in neural PCA space.

In [ ]:
# Exercise 3.2: Kernel comparison on neural data (PCA to 2D for visualization)
sc = StandardScaler()
pca = PCA(n_components=2)
X_neur2d = pca.fit_transform(sc.fit_transform(neural_rates))

kernels = {
    'Linear': SVC(kernel='linear', C=1),
    'Polynomial (d=3)': SVC(kernel='poly', degree=3, C=1, gamma='scale'),
    'RBF': SVC(kernel='rbf', C=10, gamma='scale'),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, svm) in zip(axes, kernels.items()):
    svm.fit(X_neur2d, targets)
    
    # TODO: Plot decision boundaries using meshgrid + contourf
    # TODO: Scatter training points colored by direction
    
    ax.set_title(f'{name}\nTrain acc: {svm.score(X_neur2d, targets):.1%}')

plt.suptitle('Three Kernels on Neural Direction Decoding (PCA 2D)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 3.3: LOSO accuracy by kernel (full 80D)

**Learning objective:** Compare kernel performance using the full 80-dimensional neural
data with proper LOSO cross-validation.

In [ ]:
# Exercise 3.3: LOSO accuracy by kernel
kernel_configs = {
    'Linear': SVC(kernel='linear', C=1),
    'Poly (d=2)': SVC(kernel='poly', degree=2, C=1, gamma='scale'),
    'Poly (d=3)': SVC(kernel='poly', degree=3, C=1, gamma='scale'),
    'RBF': SVC(kernel='rbf', C=10, gamma='scale'),
}

for kname, svm in kernel_configs.items():
    pipe = Pipeline([('s', StandardScaler()), ('svm', svm)])
    # TODO: Run LOSO for both direction and binary tasks
    # TODO: Print kernel name, direction accuracy, binary accuracy
    pass

---

## 🟡 Part 4: Dimensionality Revisited — VC Dimension (Lecture §4)

The lecture showed that SVMs survive high dimensions better than KNN because the
VC-dimension depends on the margin width (R²/ρ²), not on the number of features.

Here you'll replicate the noise-feature experiment from Week 8 and show that SVMs
degrade much more gracefully than KNN.

### Exercise 4.1: Noise features — SVM vs KNN vs LR vs NB (Lecture Figure 7)

**Learning objective:** Add noise features to the 6 real EMG channels and watch how
each method degrades. SVMs should survive better than KNN.

In [ ]:
# Exercise 4.1: Curse of dimensionality — SVM edition
np.random.seed(42)
noise_counts = [0, 10, 50, 100, 200, 500, 1000]

methods_noise = {
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'LR (C=10)': LogisticRegression(C=10, max_iter=2000),
    'Naive Bayes': GaussianNB(),
    'Linear SVM': SVC(kernel='linear', C=1),
}

results_noise = {m: [] for m in methods_noise}

for n_noise in noise_counts:
    noise = np.random.randn(X_raw.shape[0], n_noise)
    X_aug = np.hstack([X_raw, noise]) if n_noise > 0 else X_raw.copy()
    
    for mname, clf in methods_noise.items():
        pipe = Pipeline([('s', StandardScaler()), ('clf', clf)])
        # TODO: Run LOSO for direction decoding
        # TODO: Append mean accuracy to results_noise[mname]
        pass

# TODO: Line plot of accuracy vs total features (6 + noise) for all 4 methods
# TODO: Add chance line at 12.5%
# TODO: Use semilog x-axis

plt.show()

### 🤔 Thought Exercise
The lecture explained that the VC-dimension bound R²/ρ² does not depend on the
number of features. But adding noise features still hurts the SVM somewhat. Why?
Think about what happens to R (the data cloud radius) and ρ (the margin width) as
noise dimensions are added.

In [ ]:
# Your reflection:


---

## 🔴 Part 5: Confusion Matrices (Lecture §5)

The lecture showed that errors concentrate between adjacent directions because of
cosine tuning overlap. Here you'll build confusion matrices from LOSO predictions
and compute per-direction F1 scores.

### Exercise 5.1: Confusion matrices for four methods (Lecture Figure 8)

**Learning objective:** See which reach directions are systematically confused.
Errors should concentrate between adjacent directions (e.g., 0° ↔ 45°).

In [ ]:
# Exercise 5.1: Confusion matrices
cm_methods = {
    'Naive Bayes': GaussianNB(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Linear SVM': SVC(kernel='linear', C=1),
    'RBF SVM': SVC(kernel='rbf', C=10, gamma='scale'),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for ax, (name, clf) in zip(axes.flat, cm_methods.items()):
    pipe = Pipeline([('s', StandardScaler()), ('clf', clf)])
    
    # TODO: Collect LOSO predictions across all 20 folds
    y_pred_all = np.zeros_like(targets)
    for train_i, test_i in logo.split(neural_rates, targets, subjects):
        # TODO: Fit on training, predict on test
        pass
    
    # TODO: Build normalized confusion matrix
    # cm = confusion_matrix(targets, y_pred_all, normalize='true')
    
    # TODO: Plot with imshow, annotate cells with values
    # TODO: Set title with method name and overall LOSO accuracy
    
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices: Which Directions Are Confusable? (Neural 80D)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 5.2: Per-direction F1 scores (Lecture Figure 9)

**Learning objective:** Compute F1 per direction and see which directions are hardest
to decode. Recall: F1 combines precision ("when I say 90°, how often am I right?")
and recall ("of all actual 90° trials, how many did I catch?").

In [ ]:
# Exercise 5.2: Per-direction F1 scores
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(8)
width = 0.2

for i, (name, clf) in enumerate(cm_methods.items()):
    pipe = Pipeline([('s', StandardScaler()), ('clf', clf)])
    
    y_pred_all = np.zeros_like(targets)
    for train_i, test_i in logo.split(neural_rates, targets, subjects):
        # TODO: Fit and predict (same as 5.1)
        pass
    
    # TODO: Compute per-direction F1 using f1_score with average=None
    # TODO: Plot as grouped bar chart
    
    pass

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(dir_labels)
ax.set_xlabel('Reach Direction')
ax.set_ylabel('F1 Score (LOSO)')
ax.legend()
ax.set_ylim(0.5, 1.05)
ax.set_title('Per-Direction F1: Which Directions Are Hardest to Decode?')
plt.tight_layout()
plt.show()

---

## 🔴 Part 6: From Offline Decoding to BCI Pipeline (Lecture §6)

The lecture showed that the SVM margin provides a robustness buffer against electrode
drift. In a real BCI, the calibration model stays fixed while the incoming signals shift.
A wider margin buys more time before the drift crosses the boundary.

Here you'll simulate electrode drift and measure how each method degrades, then
compare computational costs.

### Exercise 6.1: Robustness to electrode drift (Lecture Figure 10)

**Learning objective:** Add increasing Gaussian noise to test features *without
retraining* and see which methods are most robust.

The key finding from the lecture: the RBF SVM is the *least* robust despite having the
highest clean accuracy, because its curved boundaries are more sensitive to shifts.

In [ ]:
# Exercise 6.1: Electrode drift simulation
drift_levels = np.arange(0, 2.2, 0.2)

drift_methods = {
    'LR (C=10)': LogisticRegression(C=10, max_iter=2000),
    'Naive Bayes': GaussianNB(),
    'Linear SVM': SVC(kernel='linear', C=1),
    'RBF SVM': SVC(kernel='rbf', C=10, gamma='scale'),
}

drift_results = {m: [] for m in drift_methods}

sc = StandardScaler()
X_neur_sc = sc.fit_transform(neural_rates)

for sigma in drift_levels:
    for mname, clf in drift_methods.items():
        fold_accs = []
        for train_i, test_i in logo.split(X_neur_sc, targets, subjects):
            clf.fit(X_neur_sc[train_i], targets[train_i])
            
            # TODO: Add Gaussian noise to test features (simulate drift)
            # X_test_drifted = X_neur_sc[test_i] + np.random.randn(...) * sigma
            
            # TODO: Predict on drifted test data (WITHOUT retraining)
            # TODO: Compute accuracy for this fold
            
            pass
        
        # TODO: Append mean fold accuracy to drift_results[mname]
        pass

# TODO: Line plot of accuracy vs drift level for all 4 methods
# TODO: Add chance line at 12.5%

plt.show()

### Exercise 6.2: Computational cost (Lecture Figure 11)

**Learning objective:** Measure training and prediction time for each method.
All methods are fast on our small dataset, but the scaling properties differ.

In [ ]:
# Exercise 6.2: Timing comparison
timing_methods = {
    'LR (C=10)': Pipeline([('s', StandardScaler()),
                           ('clf', LogisticRegression(C=10, max_iter=2000))]),
    'Naive Bayes': Pipeline([('s', StandardScaler()),
                             ('clf', GaussianNB())]),
    'KNN (k=5)': Pipeline([('s', StandardScaler()),
                           ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'Linear SVM': Pipeline([('s', StandardScaler()),
                            ('clf', SVC(kernel='linear', C=1))]),
    'RBF SVM': Pipeline([('s', StandardScaler()),
                         ('clf', SVC(kernel='rbf', C=10, gamma='scale'))]),
}

train_times = {}
predict_times = {}

for name, pipe in timing_methods.items():
    # TODO: Time training on full neural dataset (average over 20 runs)
    # TODO: Time prediction on full neural dataset (average over 100 runs)
    pass

# TODO: Create side-by-side bar chart: training time (left), prediction time (right)

plt.show()

### Exercise 6.3: Confidence gating with the SVM decision function

**Learning objective:** Use the SVM's decision function distance as a confidence
measure. Trials far from the boundary are confident; trials near the boundary are
uncertain.

In [ ]:
# Exercise 6.3: Decision function as confidence
# Use binary diagnosis (harder task) so confidence gating shows a real tradeoff
pipe = Pipeline([('s', StandardScaler()),
                 ('svm', SVC(kernel='linear', C=1))])

# TODO: Run LOSO on neural binary task, collecting both predictions AND decision values
# Hint: pipe.decision_function(X_test) gives signed distance from boundary
decision_vals = np.zeros(len(group_binary), dtype=float)
preds = np.zeros(len(group_binary), dtype=int)

for train_i, test_i in logo.split(neural_rates, group_binary, subjects):
    pipe.fit(neural_rates[train_i], group_binary[train_i])
    # TODO: Store decision_function values and predictions for test trials
    pass

correct = (preds == group_binary)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# TODO: Panel A — histogram of |decision_vals|, split by correct vs incorrect
# Incorrect predictions should cluster near zero (low confidence)
ax = axes[0]

# TODO: Panel B — sweep a threshold on |decision_vals|
# For each threshold, compute accuracy and coverage (fraction of trials above threshold)
# Plot accuracy vs coverage — should rise as you exclude uncertain trials
ax = axes[1]

plt.tight_layout()
plt.show()

---

## 🔴 Part 7: Bringing It Together (Lecture §7)

Compile all results into the running comparison table and reflect on when to use
each method.

### Exercise 7.1: The comparison table (Lecture Figure 12)

**Learning objective:** Update the running comparison table with SVM results.

In [ ]:
# Exercise 7.1: Running comparison table
print("Running Comparison Table Through Week 9")
print("=" * 75)
print(f"{'Week':>4} {'Method':>20} {'Features':>12} {'8-Dir':>8} {'Binary':>8} {'AUC':>6}")
print("-" * 75)

# Weeks 5-8 results (from previous labs)
prior = [
    (5, 'Logistic Reg (C=10)', 'Raw EMG (6)', '80.8%', '85.0%', '0.94'),
    (6, 'Logistic Reg (C=10)', 'Neural (80)', '91.2%', '73.3%', '0.80'),
    (6, 'Population Vector', 'Neural (80)', '95.8%', 'N/A', 'N/A'),
    (7, 'Gaussian NB', 'Raw EMG (6)', '81.2%', '65.4%', '0.68'),
    (7, 'Gaussian NB', 'Neural (80)', '95.2%', '77.7%', '0.86'),
    (8, 'KNN (k=5)', 'Raw EMG (6)', '80.2%', '100.0%', '1.00'),
    (8, 'KNN (k=5)', 'Neural (80)', '94.6%', '78.3%', '0.86'),
]

for row in prior:
    print(f"{row[0]:>4} {row[1]:>20} {row[2]:>12} {row[3]:>8} {row[4]:>8} {row[5]:>6}")

# TODO: Add Week 9 SVM results from your computations above
# Linear SVM (EMG direction, EMG binary, Neural direction, Neural binary)
# RBF SVM (same four tasks)

print("-" * 75)

### 🤔 Final Thought Exercise

The lecture showed that the Linear SVM is the most robust to electrode drift but
the RBF SVM has the highest peak accuracy. A clinician asks you to build a BCI
decoder for a patient who will use it 8 hours per day with recalibration only once
per morning. Which SVM variant would you recommend, and why?

In [ ]:
# Your reflection:


---

## Summary

This lab followed the 7 sections of the Week 9 lecture:

1. **§1 (Part 1):** Visualized the maximum-margin boundary and support vectors on neural reaching data
2. **§2 (Part 2):** Swept C, observed the margin–accuracy tradeoff, compared 5 methods on all 4 tasks
3. **§3 (Part 3):** Demonstrated the kernel trick on concentric rings and compared kernels on our data
4. **§4 (Part 4):** Showed that SVMs survive noise features better than KNN (VC-dimension argument)
5. **§5 (Part 5):** Built confusion matrices and per-direction F1 scores for 8-direction decoding
6. **§6 (Part 6):** Simulated electrode drift and measured computational cost for BCI deployment
7. **§7 (Part 7):** Updated the running comparison table — Linear SVM is the robustness champion, RBF SVM is the accuracy champion